## Test 03: rolling walk-forward backtest — train 10 years, invest 1 year

Parameters are selected by **net training Sharpe** and frozen for the next calendar year. The last partial investment year runs through the latest date in the CSV. Prior observations may warm up causal indicators, but only the preceding 10 years contribute to training performance. Earlier OOS years may enter later training windows as time advances.

The notebook checks chronological separation, causal features, delayed execution, frozen OOS parameters, future-data invariance and cost accounting. These checks target implementation leakage; they do not undo research choices informed by historical results or make revised adjusted prices a point-in-time data archive.

**Default run: RSI on the USD/UBT spread.** The first code cell sets `RULE_SET = "rsi"`; change it to `"bollinger"` to run Bollinger bands instead. Only the selected rule set is used. Generated results are saved in separate `rsi` and `bollinger` directories; existing Bollinger results are from earlier runs. Use Restart Kernel and Run All after changing the selection.

In [ ]:
PAIR = ("USD", "UBT")
SIGNAL_PAIR = PAIR
RULE_SET = "rsi"
CSV_PATH = "data/etf_daily_joint.csv"
TRAIN_MONTHS = 120
TEST_MONTHS = 12
FIRST_OOS_DATE = None
INITIAL_CAPITAL = 100_000.0
COMMISSION_BPS = 1.0
SLIPPAGE_BPS = 4.0
MIN_TRAIN_ENTRIES = 1

GRIDS = {
    "bollinger": {
        "length": [5, 10, 15, 20, 30, 40, 60],
        "ma_type": ["SMA", "EMA", "WMA"],
        "stdev": [1.0, 1.5, 2.0, 2.5, 3.0],
    },
    "rsi": {
        "rsi_window": [2, 3, 4, 5, 7, 10, 14],
        "entry_low": [5, 10, 15, 20, 25, 30],
        "entry_high": [70, 75, 80, 85, 90, 95],
        "exit_risk": [50, 60, 70, 80],
        "exit_bond": [20, 30, 40, 50],
    },
}
COST_PATHS_BPS = {
    "Zero cost": (0.0, 0.0),
    "Base cost": (COMMISSION_BPS, SLIPPAGE_BPS),
    "10 bps per leg": (1.0, 9.0),
    "20 bps per leg": (1.0, 19.0),
}

SHARPE_NULL = 0.5
SHARPE_ALPHA = 0.05
SHARPE_BOOTSTRAPS = 10_000
SHARPE_PRIMARY_BLOCK = 20
SHARPE_BLOCK_LENGTHS = [5, 10, 20, 60]
SHARPE_HAC_LAGS = 20
SHARPE_TEST_SEED = 20261028


## Load data and validate costs

Costs below are illustrative assumptions. Commission is charged on executed notional; buys execute above the quoted open and sells below it by the slippage rate. A switch incurs a sale and a purchase. Fractional shares, zero cash interest, and reinvested-distribution adjusted prices are assumed; taxes and market impact are excluded.

In [ ]:
from pathlib import Path
from itertools import product
from datetime import datetime, timezone
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import display

risk, bond = PAIR
assert risk != bond and len(SIGNAL_PAIR) == 2
assert TRAIN_MONTHS > 0 and TEST_MONTHS > 0 and INITIAL_CAPITAL > 0
source_path = Path(CSV_PATH)
prices = pd.read_csv(source_path, index_col="date", parse_dates=["date"]).sort_index()
assert prices.index.is_unique and prices.index.is_monotonic_increasing
required = list(dict.fromkeys(
    [f"{ticker}_{field}" for ticker in PAIR for field in ["open", "close", "adj_close"]]
    + [f"{ticker}_adj_close" for ticker in SIGNAL_PAIR]
))
if not set(required).issubset(prices.columns):
    raise ValueError("Selected tickers are missing; run the data-preparation notebook")
if not np.isfinite(prices[required].to_numpy()).all() or (prices[required] <= 0).any().any():
    raise ValueError("Selected prices must be complete, finite and positive; no filling is applied")
for commission, slippage in [(COMMISSION_BPS, SLIPPAGE_BPS)] + list(COST_PATHS_BPS.values()):
    if not (0 <= commission < 10_000 and 0 <= slippage < 10_000):
        raise ValueError("Commission/slippage must be between 0 and 10,000 bps")
output_dir = Path("data") / f"test_03_wfo_{risk}_{bond}" / RULE_SET / f"train_{TRAIN_MONTHS}m_test_{TEST_MONTHS}m"
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Trade {risk}/{bond}; signal {SIGNAL_PAIR[0]}/{SIGNAL_PAIR[1]}; {RULE_SET}")
print(f"Data: {prices.index[0].date()} to {prices.index[-1].date()}")
print(f"Train {TRAIN_MONTHS} months; invest {TEST_MONTHS} months; costs {COMMISSION_BPS}+{SLIPPAGE_BPS} bps per leg")


## Causal indicators and the parameter grid

Bollinger entries use the outer bands and exits use the middle band. RSI uses its four entry/exit thresholds. All decisions use the previous session’s close. There are no centered windows or future shifts.

In [ ]:
def wilder_rsi(series, window):
    window = int(window)
    if window < 1 or window >= len(series):
        raise ValueError("RSI window must be positive and shorter than the sample")
    change = series.diff()
    gains = change.clip(lower=0)
    losses = -change.clip(upper=0)
    avg_gain = gains.iloc[1:window + 1].mean()
    avg_loss = losses.iloc[1:window + 1].mean()
    values = pd.Series(np.nan, index=series.index)
    for i in range(window, len(series)):
        if i > window:
            avg_gain = ((window - 1) * avg_gain + gains.iloc[i]) / window
            avg_loss = ((window - 1) * avg_loss + losses.iloc[i]) / window
        values.iloc[i] = (50 if avg_gain == 0 and avg_loss == 0 else
                          100 if avg_loss == 0 else 100 - 100 / (1 + avg_gain / avg_loss))
    return values

def moving_average(series, length, ma_type):
    if ma_type == "SMA":
        return series.rolling(length, min_periods=length).mean()
    if ma_type == "EMA":
        return series.ewm(span=length, adjust=False, min_periods=length).mean()
    if ma_type == "WMA":
        weights = np.arange(1, length + 1, dtype=float)
        return series.rolling(length, min_periods=length).apply(
            lambda values: np.dot(values, weights) / weights.sum(), raw=True
        )
    raise ValueError(f"Unsupported MA type: {ma_type}")

def valid_parameters(parameters):
    if RULE_SET == "bollinger":
        return (
            parameters.length.ge(2) & parameters.length.lt(len(prices))
            & parameters.length.eq(parameters.length.astype(int))
            & parameters.ma_type.isin(["SMA", "EMA", "WMA"])
            & parameters.stdev.gt(0) & np.isfinite(parameters.stdev)
        )
    if RULE_SET == "rsi":
        return (
            parameters.rsi_window.ge(1) & parameters.rsi_window.lt(len(prices))
            & parameters.rsi_window.eq(parameters.rsi_window.astype(int))
            & parameters.entry_low.ge(0) & parameters.entry_high.le(100)
            & parameters.entry_low.lt(parameters.exit_bond)
            & parameters.exit_bond.le(parameters.exit_risk)
            & parameters.exit_risk.lt(parameters.entry_high)
        )
    raise ValueError("RULE_SET must be bollinger or rsi")

parameter_names = list(GRIDS[RULE_SET])
parameters = pd.DataFrame(product(*GRIDS[RULE_SET].values()), columns=parameter_names)
parameters = parameters.loc[valid_parameters(parameters)].reset_index(drop=True)
if parameters.empty:
    raise ValueError("The parameter grid is empty")

def build_features(data, parameters):
    frame = pd.DataFrame(index=data.index)
    frame["spread"] = data[f"{SIGNAL_PAIR[0]}_adj_close"] / data[f"{SIGNAL_PAIR[1]}_adj_close"]
    if RULE_SET == "bollinger":
        for length in sorted(parameters.length.unique()):
            frame[f"bb_std_{int(length)}"] = frame.spread.rolling(int(length), min_periods=int(length)).std(ddof=0)
        for length, ma_type in parameters[["length", "ma_type"]].drop_duplicates().itertuples(index=False, name=None):
            frame[f"bb_middle_{int(length)}_{ma_type}"] = moving_average(frame.spread, int(length), ma_type)
    else:
        for window in sorted(parameters.rsi_window.unique()):
            frame[f"rsi_{int(window)}"] = wilder_rsi(frame.spread, int(window))
    return frame

features = build_features(prices, parameters)


In [ ]:
def make_rules(features, parameters):
    if RULE_SET == "bollinger":
        spread_values = features.spread.to_numpy()[:, None]
        middle = features[[f"bb_middle_{int(row.length)}_{row.ma_type}" for row in parameters.itertuples()]].to_numpy()
        width = features[[f"bb_std_{int(length)}" for length in parameters.length]].to_numpy() * parameters.stdev.to_numpy()
        return {
            "enter_risk": spread_values < middle - width,
            "enter_bond": spread_values > middle + width,
            "exit_risk": spread_values > middle,
            "exit_bond": spread_values < middle,
        }
    values = features[[f"rsi_{int(window)}" for window in parameters.rsi_window]].to_numpy()
    return {
        "enter_risk": values < parameters.entry_low.to_numpy(),
        "enter_bond": values > parameters.entry_high.to_numpy(),
        "exit_risk": values > parameters.exit_risk.to_numpy(),
        "exit_bond": values < parameters.exit_bond.to_numpy(),
    }

all_rules = make_rules(features, parameters)
if np.any(all_rules["enter_risk"] & all_rules["enter_bond"]):
    raise ValueError("Conflicting entry rules")
market_close = np.column_stack([prices[f"{ticker}_adj_close"] for ticker in PAIR])
market_open = np.column_stack([
    prices[f"{ticker}_open"] * prices[f"{ticker}_adj_close"] / prices[f"{ticker}_close"]
    for ticker in PAIR
])
overnight = np.zeros((len(prices), 3))
intraday = np.zeros((len(prices), 3))
overnight[1:, 1:] = market_open[1:] / market_close[:-1] - 1
intraday[:, 1:] = market_close / market_open - 1


## Shared execution engine: prior-close signals, next-open fills, explicit transaction costs

Training starts in cash for every candidate. Across investment blocks, actual holdings carry forward; new parameters govern subsequent exits/entries without an artificial scheduled liquidation. The first OOS session starts in cash. Positions are marked to the latest close without forced final liquidation.

In [ ]:
def next_holdings(holding, signal_index, column_ids):
    updated = holding.copy()
    valid = column_ids >= 0
    updated[~valid] = 0
    if signal_index < 0 or not valid.any():
        return updated
    ids = column_ids[valid]
    active = updated[valid].copy()
    exits = ((active == 1) & all_rules["exit_risk"][signal_index, ids]) | ((active == 2) & all_rules["exit_bond"][signal_index, ids])
    active[exits] = 0
    flat = active == 0
    active[flat & all_rules["enter_risk"][signal_index, ids]] = 1
    active[flat & all_rules["enter_bond"][signal_index, ids]] = 2
    updated[valid] = active
    return updated

def execution_costs(previous, current, commission_bps, slippage_bps):
    c = commission_bps / 10_000
    s = slippage_bps / 10_000
    sell = (previous != current) & (previous != 0)
    buy = (previous != current) & (current != 0)
    after_sale = np.where(sell, (1 - s) * (1 - c), 1.0)
    factor = after_sale * np.where(buy, 1 / ((1 + s) * (1 + c)), 1.0)
    commission = np.where(sell, (1 - s) * c, 0.0) + np.where(buy, after_sale * c / (1 + c), 0.0)
    slippage = np.where(sell, s, 0.0) + np.where(buy, after_sale * s / ((1 + c) * (1 + s)), 0.0)
    return factor, commission, slippage, sell, buy

def train_grid(indices, commission_bps, slippage_bps):
    count = len(parameters)
    ids = np.arange(count)
    holding = np.zeros(count, dtype=int)
    growth = np.ones(count)
    peak = np.ones(count)
    max_dd = np.zeros(count)
    total = np.zeros(count)
    squared = np.zeros(count)
    entries = np.zeros(count, dtype=int)
    legs = np.zeros(count, dtype=int)
    for i in indices:
        previous = holding.copy()
        holding = next_holdings(holding, i - 1, ids)
        factor, _, _, sell, buy = execution_costs(previous, holding, commission_bps, slippage_bps)
        returns = (1 + overnight[i, previous]) * factor * (1 + intraday[i, holding]) - 1
        total += returns
        squared += returns ** 2
        growth *= 1 + returns
        peak = np.maximum(peak, growth)
        max_dd = np.minimum(max_dd, growth / peak - 1)
        entries += buy
        legs += sell.astype(int) + buy.astype(int)
    mean = total / len(indices)
    std = np.sqrt(np.maximum((squared - len(indices) * mean ** 2) / (len(indices) - 1), 0))
    sharpe = np.full(count, np.nan)
    np.divide(mean * np.sqrt(252), std, out=sharpe, where=std > 1e-14)
    result = parameters.copy()
    result["config_id"] = ids
    result["Train net Sharpe"] = sharpe
    result["Train net return"] = growth - 1
    result["Train max drawdown"] = max_dd
    result["Train entries"] = entries
    result["Train transaction legs"] = legs
    return result


## Rolling schedule and training-only selection

With 12-month investment windows, the default boundaries are calendar-year starts. Each training interval is the immediately preceding 120 calendar months, excluding the first OOS day. A full 10-year history must precede the first test year. If no candidate has a defined Sharpe and the required entries, that fold holds cash. The latest investment year may be partial.

In [ ]:
earliest_oos = prices.index[0] + pd.DateOffset(months=TRAIN_MONTHS)
if FIRST_OOS_DATE is None:
    if TEST_MONTHS == 12:
        first_oos = earliest_oos.to_period("Y").start_time
        if first_oos < earliest_oos:
            first_oos += pd.DateOffset(years=1)
    elif TEST_MONTHS == 3:
        first_oos = earliest_oos.to_period("Q").start_time
        if first_oos < earliest_oos:
            first_oos += pd.DateOffset(months=3)
    else:
        first_oos = earliest_oos.to_period("M").start_time
        if first_oos < earliest_oos:
            first_oos += pd.DateOffset(months=1)
else:
    first_oos = pd.Timestamp(FIRST_OOS_DATE)
if first_oos - pd.DateOffset(months=TRAIN_MONTHS) < prices.index[0]:
    raise ValueError("Insufficient history for the first full training window")
if first_oos > prices.index[-1]:
    raise ValueError("No OOS observations remain after the training window")
fold_rows = []
training_tables = []
selected_config = pd.Series(-1, index=prices.index, dtype=int)
fold_id_by_date = pd.Series(-1, index=prices.index, dtype=int)
test_start = first_oos
fold_id = 0
while test_start <= prices.index[-1]:
    test_end = test_start + pd.DateOffset(months=TEST_MONTHS)
    train_start = test_start - pd.DateOffset(months=TRAIN_MONTHS)
    train_ids = np.flatnonzero((prices.index >= train_start) & (prices.index < test_start))
    test_ids = np.flatnonzero((prices.index >= test_start) & (prices.index < test_end))
    if len(train_ids) < 2 or not len(test_ids):
        raise ValueError("Empty/insufficient training or test interval")
    candidates = train_grid(train_ids, COMMISSION_BPS, SLIPPAGE_BPS)
    training_tables.append(candidates.assign(fold=fold_id))
    eligible = candidates.loc[candidates["Train net Sharpe"].notna() & candidates["Train entries"].ge(MIN_TRAIN_ENTRIES)]
    chosen_id = -1
    details = {name: None for name in parameter_names}
    training_sharpe = np.nan
    training_entries = 0
    if not eligible.empty:
        chosen = eligible.sort_values(
            ["Train net Sharpe"] + parameter_names,
            ascending=[False] + [True] * len(parameter_names)
        ).iloc[0]
        chosen_id = int(chosen.config_id)
        details = json.loads(parameters.loc[[chosen_id]].to_json(orient="records"))[0]
        training_sharpe = float(chosen["Train net Sharpe"])
        training_entries = int(chosen["Train entries"])
    selected_config.iloc[test_ids] = chosen_id
    fold_id_by_date.iloc[test_ids] = fold_id
    fold_rows.append({
        "fold": fold_id, "train_start": train_start,
        "train_first_session": prices.index[train_ids[0]],
        "train_last_session": prices.index[train_ids[-1]],
        "test_start": test_start, "test_end_exclusive": test_end,
        "oos_first_session": prices.index[test_ids[0]],
        "oos_last_session": prices.index[test_ids[-1]],
        "train_sessions": len(train_ids), "oos_sessions": len(test_ids),
        "config_id": chosen_id, "train_net_sharpe": training_sharpe,
        "train_entries": training_entries, "cash_fallback": chosen_id < 0,
        "partial_test_block": test_end > prices.index[-1] + pd.Timedelta(days=1),
        **details,
    })
    assert prices.index[train_ids[-1]] < prices.index[test_ids[0]]
    fold_id += 1
    test_start = test_end
folds = pd.DataFrame(fold_rows).set_index("fold")
training_results = pd.concat(training_tables, ignore_index=True)
oos_ids = np.flatnonzero(fold_id_by_date.to_numpy() >= 0)
assert np.array_equal(oos_ids, np.arange(oos_ids[0], len(prices)))
print(f"{len(folds)} folds; {len(parameters)} candidates per training window")
print(f"OOS: {prices.index[oos_ids[0]].date()} to {prices.index[oos_ids[-1]].date()}")
display(folds.head(3))
display(folds.tail(3))


## Stitch one continuous OOS portfolio and record every transaction leg

In [ ]:
def simulate_oos(commission_bps, slippage_bps, keep_trades=False):
    holding = np.array([0], dtype=int)
    wealth = INITIAL_CAPITAL
    gross_wealth = INITIAL_CAPITAL
    rows = []
    trade_rows = []
    labels = ["CASH", risk, bond]
    c = commission_bps / 10_000
    s = slippage_bps / 10_000
    for i in oos_ids:
        date = prices.index[i]
        chosen_id = int(selected_config.iloc[i])
        previous = holding.copy()
        holding = next_holdings(holding, i - 1, np.array([chosen_id]))
        old, new = int(previous[0]), int(holding[0])
        factor, commission_rate, slippage_rate, sell, buy = execution_costs(previous, holding, commission_bps, slippage_bps)
        open_factor = 1 + overnight[i, old]
        intraday_factor = 1 + intraday[i, new]
        opening_wealth = wealth * open_factor
        after_costs = opening_wealth * factor[0]
        commission_dollars = opening_wealth * commission_rate[0]
        slippage_dollars = opening_wealth * slippage_rate[0]
        equity_pnl = (wealth * overnight[i, old] if old == 1 else 0.0) + (after_costs * intraday[i, new] if new == 1 else 0.0)
        bond_pnl = (wealth * overnight[i, old] if old == 2 else 0.0) + (after_costs * intraday[i, new] if new == 2 else 0.0)
        net_return = open_factor * factor[0] * intraday_factor - 1
        gross_return = open_factor * intraday_factor - 1
        new_wealth = after_costs * intraday_factor
        if keep_trades:
            remaining = opening_wealth
            if sell[0]:
                notional = remaining * (1 - s)
                trade_rows.append({"execution_date": date, "signal_date": prices.index[i - 1],
                                   "fold": int(fold_id_by_date.iloc[i]), "side": "SELL", "asset": labels[old],
                                   "executed_notional": notional, "commission": notional * c,
                                   "slippage": remaining * s})
                remaining = notional * (1 - c)
            if buy[0]:
                notional = remaining / (1 + c)
                trade_rows.append({"execution_date": date, "signal_date": prices.index[i - 1],
                                   "fold": int(fold_id_by_date.iloc[i]), "side": "BUY", "asset": labels[new],
                                   "executed_notional": notional, "commission": notional * c,
                                   "slippage": notional * s / (1 + s)})
        gross_wealth *= 1 + gross_return
        rows.append({
            "date": date, "fold": int(fold_id_by_date.iloc[i]), "config_id": chosen_id,
            "signal_date": prices.index[i - 1], "previous_position": labels[old], "position": labels[new],
            "starting_equity": wealth, "net_return": net_return, "gross_return": gross_return,
            "net_equity": new_wealth, "gross_equity": gross_wealth,
            "equity_pnl": equity_pnl, "bond_pnl": bond_pnl,
            "commission": commission_dollars, "slippage": slippage_dollars,
            "transaction_legs": int(sell[0]) + int(buy[0]),
        })
        wealth = new_wealth
    daily = pd.DataFrame(rows).set_index("date")
    np.testing.assert_allclose(daily.equity_pnl + daily.bond_pnl - daily.commission - daily.slippage,
                               daily.net_equity - daily.starting_equity, rtol=1e-9, atol=1e-7)
    trade_columns = ["execution_date", "signal_date", "fold", "side", "asset", "executed_notional", "commission", "slippage"]
    ledger = pd.DataFrame(trade_rows, columns=trade_columns)
    if not ledger.empty:
        assert (ledger.signal_date < ledger.execution_date).all()
        np.testing.assert_allclose(ledger.commission.sum(), daily.commission.sum(), rtol=1e-10)
        np.testing.assert_allclose(ledger.slippage.sum(), daily.slippage.sum(), rtol=1e-10)
    return daily, ledger

daily, trades = simulate_oos(COMMISSION_BPS, SLIPPAGE_BPS, keep_trades=True)
for fold, block in daily.groupby("fold", sort=True):
    r = block.net_return
    folds.loc[fold, "oos_net_return"] = (1 + r).prod() - 1
    folds.loc[fold, "oos_net_sharpe"] = r.mean() / r.std() * np.sqrt(252) if r.std() > 0 else np.nan
    folds.loc[fold, "oos_cost_paid"] = (block.commission + block.slippage).sum()
display(trades.head(10))


## Performance and cost-path sensitivity

Every cost path uses exactly the same parameters and holding decisions selected under the base-cost training assumption. They isolate cost sensitivity; parameters are not re-optimized for each path. Gross-minus-net wealth includes compounding effects and differs from cash costs paid.

In [ ]:
def performance(returns):
    growth = (1 + returns).cumprod()
    dd = growth / growth.cummax().clip(lower=1) - 1
    years = (returns.index[-1] - prices.index[oos_ids[0] - 1]).days / 365.25
    vol = returns.std() * np.sqrt(252)
    downside = np.sqrt(returns.clip(upper=0).pow(2).mean()) * np.sqrt(252)
    return {"CAGR": growth.iloc[-1] ** (1 / years) - 1,
            "Sharpe": returns.mean() * 252 / vol if vol > 0 else np.nan,
            "Sortino": returns.mean() * 252 / downside if downside > 0 else np.nan,
            "Annualized volatility": vol, "Max drawdown": dd.min(),
            "Total return": growth.iloc[-1] - 1}

cost_daily = {}
for name, (commission, slippage) in COST_PATHS_BPS.items():
    cost_daily[name], _ = simulate_oos(commission, slippage)
cost_equities = pd.DataFrame({name: frame.net_equity for name, frame in cost_daily.items()})
cost_metrics = pd.DataFrame({name: performance(frame.net_return) for name, frame in cost_daily.items()}).T
assessment = pd.DataFrame({"WFO net": performance(daily.net_return), "Same decisions, zero cost": performance(daily.gross_return)}).T
formats = {key: "{:.2%}" for key in ["CAGR", "Annualized volatility", "Max drawdown", "Total return"]}
formats.update({"Sharpe": "{:.3f}", "Sortino": "{:.3f}"})
display(assessment.style.format(formats))
display(cost_metrics.style.format(formats))
print(f"Transaction legs: {daily.transaction_legs.sum():,}; commission paid: {daily.commission.sum():,.2f}; slippage paid: {daily.slippage.sum():,.2f}")


## OOS equity and drawdowns; buy-and-hold benchmarks enter at the first OOS open, including base entry costs

In [ ]:
buy_factor = 1 / ((1 + SLIPPAGE_BPS / 10_000) * (1 + COMMISSION_BPS / 10_000))
curves = pd.DataFrame({"WFO net": daily.net_equity, "WFO zero cost, same decisions": daily.gross_equity})
for j, asset in enumerate(PAIR):
    curves[f"{asset} buy and hold, net entry"] = INITIAL_CAPITAL * buy_factor * market_close[oos_ids, j] / market_open[oos_ids[0], j]
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True,
                         gridspec_kw={"height_ratios": [3, 1]}, constrained_layout=True)
curves.plot(ax=axes[0], logy=True, grid=True)
axes[0].set_title(f"{risk}/{bond} {RULE_SET}: rolling {TRAIN_MONTHS}m train / {TEST_MONTHS}m OOS")
axes[0].set_ylabel("Portfolio value")
for name, line in zip(curves.columns, axes[0].lines):
    dd = curves[name] / curves[name].cummax().clip(lower=INITIAL_CAPITAL) - 1
    axes[1].plot(curves.index, dd, color=line.get_color(), linewidth=1)
axes[1].set_ylabel("Drawdown")
axes[1].yaxis.set_major_formatter(PercentFormatter(1))
axes[1].grid(alpha=0.3)
fig.savefig(output_dir / "oos_equity_drawdown.png", dpi=140)
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True, constrained_layout=True)
cost_equities.plot(ax=axes[0], logy=True, grid=True)
axes[0].set_title("OOS trading-cost paths: same selected parameters and holding decisions")
axes[0].set_ylabel("Portfolio value")
pd.DataFrame({"Commission paid": daily.commission.cumsum(),
              "Slippage paid": daily.slippage.cumsum(),
              "Wealth drag: zero cost minus net": daily.gross_equity - daily.net_equity}).plot(ax=axes[1], grid=True)
axes[1].set_ylabel("Cumulative amount")
fig.savefig(output_dir / "trading_cost_paths.png", dpi=140)
plt.show()


## Monthly OOS net returns and compounded yearly totals; boundary years may be partial

In [ ]:
monthly = (1 + daily.net_return).resample("ME").prod(min_count=1) - 1
yearly = (1 + monthly).groupby(monthly.index.year).prod(min_count=1) - 1
monthly_table = pd.DataFrame({"year": monthly.index.year, "month": monthly.index.month,
                              "return": monthly.to_numpy()}).pivot(index="year", columns="month", values="return")
monthly_table = monthly_table.reindex(columns=range(1, 13))
monthly_table["Year"] = yearly
values = monthly_table.to_numpy()
limit = max(float(np.nanmax(np.abs(values))), 0.001)
fig, ax = plt.subplots(figsize=(15, 8), constrained_layout=True)
image = ax.imshow(np.ma.masked_invalid(values), cmap="RdYlGn", aspect="auto", vmin=-limit, vmax=limit)
ax.set_xticks(range(13), ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec", "Year"])
ax.set_yticks(range(len(monthly_table)), monthly_table.index)
ax.axvline(11.5, color="black", linewidth=1)
for row in range(len(monthly_table)):
    for col in range(13):
        value = values[row, col]
        if np.isfinite(value):
            ax.text(col, row, f"{value:.1%}", ha="center", va="center", fontsize=8,
                    color="white" if abs(value) > 0.65 * limit else "black")
fig.colorbar(image, ax=ax).ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.set_title(f"{risk}/{bond} {RULE_SET}: walk-forward OOS NET monthly/yearly returns")
fig.savefig(output_dir / "oos_monthly_returns.png", dpi=140)
plt.show()


## Return attribution protocol: equities, bonds and explicit negative cost contributions

In [ ]:
pnl = pd.DataFrame({"Equity": daily.equity_pnl, "Bonds": daily.bond_pnl,
                    "Commission": -daily.commission, "Slippage": -daily.slippage})
contributions = {}
for label, frequency in [("monthly", "ME"), ("yearly", "YE")]:
    table = pnl.resample(frequency).sum().div(daily.starting_equity.resample(frequency).first(), axis=0)
    table["Total"] = (1 + daily.net_return).resample(frequency).prod() - 1
    np.testing.assert_allclose(table.drop(columns="Total").sum(axis=1), table.Total, rtol=1e-9, atol=1e-12)
    contributions[label] = table
annual = contributions["yearly"]
fig, ax = plt.subplots(figsize=(15, 6), constrained_layout=True)
x = np.arange(len(annual))
positive = np.zeros(len(annual))
negative = np.zeros(len(annual))
for name, color in [("Equity", "tab:blue"), ("Bonds", "tab:orange"), ("Commission", "tab:red"), ("Slippage", "tab:purple")]:
    value = annual[name].to_numpy()
    ax.bar(x, value, bottom=np.where(value >= 0, positive, negative), label=name, color=color)
    positive += np.maximum(value, 0)
    negative += np.minimum(value, 0)
ax.scatter(x, annual.Total, color="black", marker="D", label="Net strategy return", s=20, zorder=3)
ax.set_xticks(x, annual.index.year, rotation=60)
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.set_ylabel("Contribution (percentage points)")
ax.set_title(f"{risk}/{bond} {RULE_SET}: annual OOS return contributions")
ax.axhline(0, color="gray", linewidth=0.7)
ax.grid(axis="y", alpha=0.2)
ax.legend()
fig.savefig(output_dir / "oos_return_contributions.png", dpi=140)
plt.show()


## Selected parameters by investment year

In [ ]:
fig, axes = plt.subplots(len(parameter_names), 1, figsize=(14, 2.1 * len(parameter_names)), sharex=True, constrained_layout=True)
for name, ax in zip(parameter_names, np.atleast_1d(axes)):
    if name == "ma_type":
        labels = ["SMA", "EMA", "WMA"]
        ax.step(folds.oos_first_session, folds[name].map({value: i for i, value in enumerate(labels)}), where="post")
        ax.set_yticks(range(len(labels)), labels)
    else:
        ax.step(folds.oos_first_session, folds[name], where="post")
    ax.set_ylabel(name)
    ax.grid(alpha=0.3)
fig.suptitle("Parameters frozen within each OOS investment block")
fig.savefig(output_dir / "selected_parameters.png", dpi=140)
plt.show()


## Leakage, look-ahead and accounting audit

These tests replay completed training windows and earlier OOS results after changing future prices. They also remove future observations when recomputing features and test synthetic execution timing. A failure stops the notebook before CSV export.

In [ ]:
audit_rows = []

def record_check(name, passed=True):
    if not bool(passed):
        raise AssertionError(name)
    audit_rows.append({"check": name, "status": "PASS"})

record_check("Every fold trains strictly before its first OOS session",
             (folds.train_last_session < folds.oos_first_session).all())
record_check("All OOS sessions occur exactly once with no date gaps",
             daily.index.is_unique and daily.index.equals(prices.index[oos_ids]))
record_check("Each trade executes after the signal date",
             (daily.signal_date < daily.index).all()
             and (trades.empty or (trades.signal_date < trades.execution_date).all()))
pd.testing.assert_series_equal(daily.previous_position,
                               daily.position.shift(1, fill_value="CASH"), check_names=False)
record_check("Actual OOS holdings carry continuously across annual boundaries")

for fold, row in folds.iterrows():
    train_ids = np.flatnonzero((prices.index >= row.train_start) & (prices.index < row.test_start))
    test_ids = np.flatnonzero((prices.index >= row.test_start) & (prices.index < row.test_end_exclusive))
    assert row.train_start == row.test_start - pd.DateOffset(months=TRAIN_MONTHS)
    assert row.test_end_exclusive == row.test_start + pd.DateOffset(months=TEST_MONTHS)
    assert len(np.intersect1d(train_ids, test_ids)) == 0
    assert row.train_sessions == len(train_ids) and row.oos_sessions == len(test_ids)
    block = daily.loc[daily.fold == fold]
    assert block.config_id.nunique() == 1 and block.config_id.iloc[0] == row.config_id
    candidates = training_results.loc[training_results.fold == fold]
    assert len(candidates) == len(parameters)
    eligible = candidates.loc[candidates["Train net Sharpe"].notna()
                              & candidates["Train entries"].ge(MIN_TRAIN_ENTRIES)]
    if eligible.empty:
        assert row.config_id == -1
    else:
        winner = eligible.sort_values(["Train net Sharpe"] + parameter_names,
                                       ascending=[False] + [True] * len(parameter_names)).iloc[0]
        assert int(winner.config_id) == int(row.config_id)
        np.testing.assert_allclose(row.train_net_sharpe, winner["Train net Sharpe"], rtol=1e-12)
        for name in parameter_names:
            assert row[name] == parameters.loc[int(row.config_id), name]
record_check("Exact rolling training/test windows are disjoint within every fold")
record_check("Selected parameters maximize eligible training-only net Sharpe")
record_check("Selected configuration stays frozen throughout each OOS investment block")

for fold in sorted(set([int(folds.index[0]), int(folds.index[-1])])):
    cutoff = folds.loc[fold, "train_last_session"]
    prefix = prices.loc[:cutoff]
    prefix_features = build_features(prefix, parameters)
    pd.testing.assert_frame_equal(prefix_features, features.loc[:cutoff], rtol=1e-12, atol=1e-12)
record_check("Indicators match when all observations after training cutoff are removed")

saved_rules = all_rules
try:
    all_rules = {name: np.zeros_like(mask) for name, mask in saved_rules.items()}
    i = int(oos_ids[0])
    all_rules["enter_risk"][i, 0] = True
    all_rules["exit_risk"][i + 1, 0] = True
    assert next_holdings(np.array([0]), i - 1, np.array([0]))[0] == 0
    assert next_holdings(np.array([0]), i, np.array([0]))[0] == 1
    assert next_holdings(np.array([1]), i, np.array([0]))[0] == 1
    assert next_holdings(np.array([1]), i + 1, np.array([0]))[0] == 0
finally:
    all_rules = saved_rules
record_check("Synthetic entries and exits cannot execute using the current session close")

saved_prices = prices
saved_rules = all_rules
saved_overnight = overnight
saved_intraday = intraday
perturbation_cutoff = folds.iloc[0].test_end_exclusive
try:
    prices = saved_prices.copy()
    future = prices.index >= perturbation_cutoff
    if future.any():
        multipliers = np.linspace(1.35, 2.0, int(future.sum()))
        for field in ["open", "close", "adj_close"]:
            column = f"{SIGNAL_PAIR[0]}_{field}"
            if column in prices:
                prices.loc[future, column] = prices.loc[future, column].to_numpy() * multipliers
        changed_features = build_features(prices, parameters)
        pd.testing.assert_frame_equal(changed_features.loc[changed_features.index < perturbation_cutoff],
                                       features.loc[features.index < perturbation_cutoff], rtol=1e-12, atol=1e-12)
        all_rules = make_rules(changed_features, parameters)
        changed_close = np.column_stack([prices[f"{ticker}_adj_close"] for ticker in PAIR])
        changed_open = np.column_stack([
            prices[f"{ticker}_open"] * prices[f"{ticker}_adj_close"] / prices[f"{ticker}_close"]
            for ticker in PAIR
        ])
        overnight = np.zeros_like(saved_overnight)
        intraday = np.zeros_like(saved_intraday)
        overnight[1:, 1:] = changed_open[1:] / changed_close[:-1] - 1
        intraday[:, 1:] = changed_close / changed_open - 1
        earlier_folds = folds.loc[folds.train_last_session < perturbation_cutoff]
        for fold, row in earlier_folds.iterrows():
            train_ids = np.flatnonzero((prices.index >= row.train_start) & (prices.index < row.test_start))
            replay = train_grid(train_ids, COMMISSION_BPS, SLIPPAGE_BPS)
            original = training_results.loc[training_results.fold == fold].drop(columns="fold").reset_index(drop=True)
            pd.testing.assert_frame_equal(replay.reset_index(drop=True), original, rtol=1e-12, atol=1e-12)
        changed_daily, _ = simulate_oos(COMMISSION_BPS, SLIPPAGE_BPS)
        pd.testing.assert_frame_equal(changed_daily.loc[changed_daily.index < perturbation_cutoff],
                                       daily.loc[daily.index < perturbation_cutoff], rtol=1e-12, atol=1e-10)
        record_check("Changing future prices cannot alter earlier training results or OOS trades/equity")
    else:
        audit_rows.append({"check": "Future-price perturbation requires data after the first test block",
                           "status": "NOT APPLICABLE"})
finally:
    prices = saved_prices
    all_rules = saved_rules
    overnight = saved_overnight
    intraday = saved_intraday

zero_cost, _ = simulate_oos(0.0, 0.0)
np.testing.assert_allclose(zero_cost.net_equity, daily.gross_equity, rtol=1e-11)
record_check("Zero-cost replay equals the gross curve with identical holding decisions")
factor, commission_rate, slippage_rate, sells, buys = execution_costs(
    np.array([0, 1, 1, 1]), np.array([1, 0, 2, 1]), COMMISSION_BPS, SLIPPAGE_BPS
)
np.testing.assert_allclose(factor + commission_rate + slippage_rate, 1.0, atol=1e-15)
assert (sells.astype(int) + buys.astype(int)).tolist() == [1, 1, 2, 0]
np.testing.assert_allclose(trades.commission.sum(), daily.commission.sum(), rtol=1e-10)
np.testing.assert_allclose(trades.slippage.sum(), daily.slippage.sum(), rtol=1e-10)
np.testing.assert_allclose(daily.equity_pnl + daily.bond_pnl - daily.commission - daily.slippage,
                           daily.net_equity - daily.starting_equity, rtol=1e-9, atol=1e-7)
record_check("Entries/exits pay one cost leg, switches pay two, unchanged holdings pay none")
record_check("Trade-ledger costs and equity/bond/cost attribution reconcile to net equity")
leakage_audit = pd.DataFrame(audit_rows)
display(leakage_audit)
print(f"{leakage_audit.status.eq('PASS').sum()} audit checks passed")


## Historical OOS Sharpe hypothesis test: H0 ≤ 0.5 versus H1 > 0.5

Tests the **underlying** Sharpe using the stitched historical daily OOS net returns, under the same zero-risk-free and √252 reporting convention. It does not pool simulated paths or average annual Sharpes. The primary block length is pre-specified; all alternatives are reported rather than selecting a favorable p-value.

A stationary block bootstrap resamples the original returns. Its pivot is `(bootstrap Sharpe − observed Sharpe) / bootstrap HAC standard error`. Compare that reference distribution with `(observed Sharpe − null boundary) / observed HAC standard error`. The one-sided lower confidence bound is `observed Sharpe − bootstrap upper critical value × observed standard error`. **This does not use the zero-mean null construction.**

Inference is approximate under weak dependence, sufficiently stable return behavior and adequate moments. It is conditional on the chosen WFO design and does not correct for our earlier searches across indicators, pairs or window lengths. A p-value is not the probability that the null hypothesis is true. See [Ledoit and Wolf](https://www.ledoit.net/Robust_Sharpe_2008.pdf).

In [ ]:
from numba import njit

@njit
def sharpe_hac_stat(values, lags):
    n = len(values)
    mean = values.mean()
    variance = 0.0
    for value in values:
        variance += (value - mean) ** 2
    variance /= n
    if variance <= 1e-28:
        return np.nan, np.nan
    sigma = np.sqrt(variance)
    correction = np.sqrt((n - 1.0) / n)
    scale = np.sqrt(252.0) * correction
    sharpe = scale * mean / sigma
    influence = np.empty(n)
    for i in range(n):
        centered = values[i] - mean
        influence[i] = scale * (centered / sigma - mean / (2 * sigma ** 3) * (centered ** 2 - variance))
    influence -= influence.mean()
    long_run_variance = np.dot(influence, influence) / n
    bandwidth = min(lags, n - 1)
    for lag in range(1, bandwidth + 1):
        weight = 1 - lag / (bandwidth + 1.0)
        covariance = np.dot(influence[lag:], influence[:-lag]) / n
        long_run_variance += 2 * weight * covariance
    standard_error = np.sqrt(max(long_run_variance, 0.0) / n)
    return sharpe, standard_error

@njit
def studentized_sharpe_bootstrap(values, replicates, block_length, lags, seed):
    np.random.seed(seed)
    n = len(values)
    observed, observed_se = sharpe_hac_stat(values, lags)
    output = np.empty((replicates, 3))
    sampled = np.empty(n)
    accepted = 0
    attempts = 0
    while accepted < replicates and attempts < replicates * 20:
        attempts += 1
        index = 0
        for i in range(n):
            if i == 0 or np.random.random() < 1.0 / block_length:
                index = np.random.randint(0, n)
            else:
                index = (index + 1) % n
            sampled[i] = values[index]
        estimate, se = sharpe_hac_stat(sampled, lags)
        if np.isfinite(estimate) and np.isfinite(se) and se > 1e-14:
            output[accepted, 0] = estimate
            output[accepted, 1] = se
            output[accepted, 2] = (estimate - observed) / se
            accepted += 1
    return output[:accepted], attempts


In [ ]:
if not (0 < SHARPE_ALPHA < 1 and SHARPE_BOOTSTRAPS >= 1000 and SHARPE_HAC_LAGS >= 0):
    raise ValueError("Invalid Sharpe-test settings")
if SHARPE_PRIMARY_BLOCK not in SHARPE_BLOCK_LENGTHS:
    raise ValueError("The pre-specified primary block must be in the sensitivity list")
if len(set(SHARPE_BLOCK_LENGTHS)) != len(SHARPE_BLOCK_LENGTHS) or min(SHARPE_BLOCK_LENGTHS) < 1:
    raise ValueError("Bootstrap block lengths must be unique positive values")
if "simulation_year" in daily.columns or "simulation_session" in daily.columns:
    raise ValueError("This test requires historical OOS observations, not simulated years")
sharpe_test_returns = np.ascontiguousarray(daily.net_return.to_numpy(dtype=float))
if len(sharpe_test_returns) < 30 or not np.isfinite(sharpe_test_returns).all():
    raise ValueError("Historical OOS returns must be finite with at least 30 observations")
returns_before_test = sharpe_test_returns.copy()
observed_sharpe, observed_se = sharpe_hac_stat(sharpe_test_returns, SHARPE_HAC_LAGS)
if not np.isfinite(observed_se) or observed_se <= 1e-14:
    raise ValueError("Sharpe inference is undefined for this return series")
np.testing.assert_allclose(observed_sharpe, assessment.loc["WFO net", "Sharpe"], rtol=1e-12)
observed_t = (observed_sharpe - SHARPE_NULL) / observed_se
block_seeds = [int(seed.generate_state(1)[0]) for seed in np.random.SeedSequence(SHARPE_TEST_SEED).spawn(len(SHARPE_BLOCK_LENGTHS))]
sharpe_test_rows = []
sharpe_bootstrap_tables = []
for block, seed in zip(SHARPE_BLOCK_LENGTHS, block_seeds):
    draws, attempts = studentized_sharpe_bootstrap(
        sharpe_test_returns, SHARPE_BOOTSTRAPS, block, SHARPE_HAC_LAGS, seed
    )
    if len(draws) != SHARPE_BOOTSTRAPS:
        raise ValueError("Too many degenerate bootstrap samples; Sharpe inference is unreliable")
    pivots = draws[:, 2]
    p_value = (1 + np.count_nonzero(pivots >= observed_t)) / (len(pivots) + 1)
    critical = float(np.quantile(pivots, 1 - SHARPE_ALPHA, method="higher"))
    lower_bound = observed_sharpe - critical * observed_se
    sharpe_test_rows.append({
        "block_length": block, "primary": block == SHARPE_PRIMARY_BLOCK,
        "observed_sharpe": observed_sharpe, "null_sharpe": SHARPE_NULL,
        "HAC_standard_error": observed_se, "observed_t": observed_t,
        "one_sided_p_value": p_value, "one_sided_lower_bound": lower_bound,
        "confidence_level": 1 - SHARPE_ALPHA, "reject_null": p_value < SHARPE_ALPHA,
        "replicates": len(draws), "discarded_degenerate_draws": attempts - len(draws),
        "seed": seed,
    })
    table = pd.DataFrame(draws, columns=["bootstrap_sharpe", "bootstrap_HAC_se", "studentized_pivot"])
    table["block_length"] = block
    table["replicate"] = np.arange(len(draws))
    sharpe_bootstrap_tables.append(table)
    print(f"Block {block}: p={p_value:.4f}; lower bound={lower_bound:.3f}", flush=True)
sharpe_test = pd.DataFrame(sharpe_test_rows).set_index("block_length")
sharpe_bootstrap = pd.concat(sharpe_bootstrap_tables, ignore_index=True)
primary_sharpe_test = sharpe_test.loc[SHARPE_PRIMARY_BLOCK]
display(sharpe_test.style.format({
    "observed_sharpe": "{:.3f}", "null_sharpe": "{:.2f}", "HAC_standard_error": "{:.3f}",
    "observed_t": "{:.3f}", "one_sided_p_value": "{:.4f}", "one_sided_lower_bound": "{:.3f}",
    "confidence_level": "{:.0%}",
}))
print(f"Historical OOS: {daily.index[0].date()} to {daily.index[-1].date()}, {len(daily):,} daily observations")
print(f"H0: underlying OOS Sharpe <= {SHARPE_NULL}; H1: Sharpe > {SHARPE_NULL}")
if primary_sharpe_test.reject_null:
    print(f"PRIMARY TEST: reject H0 at alpha={SHARPE_ALPHA}; evidence supports Sharpe above {SHARPE_NULL} under this fixed design.")
else:
    print(f"PRIMARY TEST: do not reject H0 at alpha={SHARPE_ALPHA}; insufficient evidence that Sharpe exceeds {SHARPE_NULL}.")


In [ ]:
primary_pivots = sharpe_bootstrap.loc[sharpe_bootstrap.block_length == SHARPE_PRIMARY_BLOCK, "studentized_pivot"].to_numpy()
critical = float(np.quantile(primary_pivots, 1 - SHARPE_ALPHA, method="higher"))
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
axes[0].hist(primary_pivots, bins=60, density=True, alpha=.7, label="Centered/studentized bootstrap reference")
axes[0].axvline(observed_t, color="red", label=f"Observed statistic: {observed_t:.2f}")
axes[0].axvline(critical, color="black", linestyle="--", label=f"{1 - SHARPE_ALPHA:.0%} critical value")
axes[0].set_xlabel("Studentized statistic")
axes[0].set_title(f"Primary block {SHARPE_PRIMARY_BLOCK}; one-sided p={primary_sharpe_test.one_sided_p_value:.4f}")
axes[0].legend(fontsize=8)
positions = np.arange(len(sharpe_test))
axes[1].plot(positions, sharpe_test.one_sided_lower_bound, marker="o", label=f"One-sided {1 - SHARPE_ALPHA:.0%} lower bound")
axes[1].axhline(SHARPE_NULL, color="red", linestyle="--", label=f"Null boundary {SHARPE_NULL}")
axes[1].axhline(observed_sharpe, color="gray", linestyle=":", label=f"Observed Sharpe {observed_sharpe:.3f}")
axes[1].set_xticks(positions, sharpe_test.index)
axes[1].set_xlabel("Mean bootstrap block length (sessions)")
axes[1].set_ylabel("Reported annualized Sharpe")
axes[1].set_title("Block-length sensitivity; primary length fixed before testing")
axes[1].legend(fontsize=8)
for ax in axes: ax.grid(alpha=.2)
fig.suptitle(f"Historical OOS net Sharpe test: H0 <= {SHARPE_NULL}; zero risk-free rate")
fig.savefig(output_dir / "oos_sharpe_hypothesis_test.png", dpi=140)
plt.show()


## Sharpe-inference checks

In [ ]:
sharpe_audit_rows = []
np.testing.assert_allclose(observed_sharpe, sharpe_test_returns.mean() / sharpe_test_returns.std(ddof=1) * np.sqrt(252), rtol=1e-12)
sharpe_audit_rows.append({"check": "Test Sharpe matches reported historical net OOS Sharpe", "status": "PASS"})
x = sharpe_test_returns
n = len(x)
m = x.mean()
variance = ((x - m) ** 2).mean()
sigma = np.sqrt(variance)
scale = np.sqrt(252) * np.sqrt((n - 1) / n)
influence = scale * ((x - m) / sigma - m / (2 * sigma ** 3) * ((x - m) ** 2 - variance))
influence -= influence.mean()
bandwidth = min(SHARPE_HAC_LAGS, n - 1)
lrv = np.dot(influence, influence) / n
for lag in range(1, bandwidth + 1):
    lrv += 2 * (1 - lag / (bandwidth + 1)) * np.dot(influence[lag:], influence[:-lag]) / n
np.testing.assert_allclose(observed_se, np.sqrt(max(lrv, 0) / n), rtol=1e-10)
sharpe_audit_rows.append({"check": "Compiled HAC standard error matches independent NumPy calculation", "status": "PASS"})
scaled_sharpe, scaled_se = sharpe_hac_stat(x * 10, SHARPE_HAC_LAGS)
np.testing.assert_allclose([scaled_sharpe, scaled_se], [observed_sharpe, observed_se], rtol=1e-10)
sharpe_audit_rows.append({"check": "Positive rescaling leaves Sharpe and inference unchanged", "status": "PASS"})
primary_seed = int(primary_sharpe_test.seed)
replay, _ = studentized_sharpe_bootstrap(x, 64, SHARPE_PRIMARY_BLOCK, SHARPE_HAC_LAGS, primary_seed)
original = sharpe_bootstrap.loc[sharpe_bootstrap.block_length == SHARPE_PRIMARY_BLOCK,
                                ["bootstrap_sharpe", "bootstrap_HAC_se", "studentized_pivot"]].iloc[:64].to_numpy()
np.testing.assert_allclose(replay, original, rtol=1e-12)
sharpe_audit_rows.append({"check": "Fixed bootstrap seed reproduces the same draws", "status": "PASS"})
thresholds = np.array([-0.5, 0, 0.5, 1, 1.5])
p_values = [(1 + np.count_nonzero(primary_pivots >= (observed_sharpe - threshold) / observed_se)) / (len(primary_pivots) + 1)
            for threshold in thresholds]
assert (np.diff(p_values) >= 0).all()
sharpe_audit_rows.append({"check": "A higher Sharpe hurdle cannot lower the one-sided p-value", "status": "PASS"})
for _, row in sharpe_test.iterrows():
    if abs(row.one_sided_p_value - SHARPE_ALPHA) > 2 / (row.replicates + 1):
        assert bool(row.reject_null) == bool(row.one_sided_lower_bound > SHARPE_NULL)
sharpe_audit_rows.append({"check": "P-value and one-sided confidence-bound decisions agree outside Monte Carlo boundary", "status": "PASS"})
np.testing.assert_array_equal(sharpe_test_returns, returns_before_test)
sharpe_audit_rows.append({"check": "Inference does not mutate or add observations to the historical return series", "status": "PASS"})
sharpe_test_audit = pd.DataFrame(sharpe_audit_rows)
display(sharpe_test_audit)


## Export the full audit trail and verify OOS accounting

In [ ]:
assert (folds.train_last_session < folds.oos_first_session).all()
assert daily.index.is_unique and daily.index.is_monotonic_increasing
assert (daily.signal_date < daily.index).all()
np.testing.assert_allclose((1 + daily.net_return).cumprod() * INITIAL_CAPITAL, daily.net_equity, rtol=1e-10)
np.testing.assert_allclose((1 + monthly).prod(), daily.net_equity.iloc[-1] / INITIAL_CAPITAL, rtol=1e-10)
np.testing.assert_allclose(daily.transaction_legs.sum(), len(trades))
for name, frame in [("sharpe_hypothesis_test", sharpe_test), ("sharpe_test_audit", sharpe_test_audit), ("oos_daily", daily), ("trades", trades), ("folds", folds), ("leakage_audit", leakage_audit),
                    ("training_grid_results", training_results), ("assessment", assessment),
                    ("cost_path_equities", cost_equities), ("cost_path_metrics", cost_metrics),
                    ("monthly_returns", monthly_table),
                    ("monthly_contributions", contributions["monthly"]),
                    ("yearly_contributions", contributions["yearly"])]:
    frame.to_csv(output_dir / f"{name}.csv", index=True)
sharpe_bootstrap.to_csv(output_dir / "sharpe_bootstrap_draws.csv.gz", index=False, compression="gzip")
manifest = {
    "pair": list(PAIR), "signal_pair": list(SIGNAL_PAIR), "rule_set": RULE_SET,
    "train_months": TRAIN_MONTHS, "test_months": TEST_MONTHS,
    "audit_checks_passed": int(leakage_audit.status.eq("PASS").sum()),
    "audit_failures": 0,
    "sharpe_hypothesis_test": {
        "null": SHARPE_NULL, "alternative": "underlying historical OOS net Sharpe > null",
        "alpha": SHARPE_ALPHA, "primary_block": SHARPE_PRIMARY_BLOCK,
        "block_lengths": SHARPE_BLOCK_LENGTHS, "HAC_lags": SHARPE_HAC_LAGS,
        "replicates_per_block": SHARPE_BOOTSTRAPS, "seed": SHARPE_TEST_SEED,
        "method": "Studentized stationary block bootstrap; Sharpe influence-function Newey-West HAC standard errors",
        "risk_free_rate": 0, "reported_annualization": 252,
        "observed_sharpe": float(observed_sharpe),
        "primary_p_value": float(primary_sharpe_test.one_sided_p_value),
        "primary_lower_bound": float(primary_sharpe_test.one_sided_lower_bound),
        "reject_null": bool(primary_sharpe_test.reject_null),
        "selection_adjusted": False, "synthetic_observations_used": 0,
    },
    "audit_limitations": "Does not remove historical research-selection bias or provider revisions to adjusted data",
    "grid": GRIDS[RULE_SET], "selection": "Maximum training net Sharpe; 252 sessions/year; zero risk-free rate",
    "min_train_entries": MIN_TRAIN_ENTRIES, "initial_capital": INITIAL_CAPITAL,
    "commission_bps": COMMISSION_BPS, "slippage_bps": SLIPPAGE_BPS,
    "cost_paths_bps": COST_PATHS_BPS, "cost_paths_reoptimized": False,
    "oos_start": daily.index[0].date().isoformat(), "oos_end": daily.index[-1].date().isoformat(),
    "folds": len(folds), "cash_fallback_folds": int(folds.cash_fallback.sum()),
    "boundary_policy": "Carry actual OOS holding; apply new parameters using previous-close signal at next open",
    "training_start_state": "cash", "oos_initial_state": "cash", "force_final_liquidation": False,
    "source_csv": str(source_path), "source_sha256": hashlib.sha256(source_path.read_bytes()).hexdigest(),
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
}
(output_dir / "run_config.json").write_text(json.dumps(manifest, indent=2) + "\n")
print(f"Verified and saved all results to {output_dir.resolve()}")
